# Fondasi Machine Learning

## Bagian 2d: Regularisasi dan Gradient Descent

Bagian ini berisi latihan bertahap: regresi pada data sinusoidal yang bising, pengaruh regularisasi pada koefisien, lalu perbandingan Ridge, Lasso, ElasticNet, dan Stochastic Gradient Descent pada data harga rumah.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import (
    ElasticNetCV,
    Lasso,
    LassoCV,
    LinearRegression,
    Ridge,
    RidgeCV,
    SGDRegressor,
)
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures

warnings.filterwarnings("ignore")
sns.set_style("white")
sns.set_context("talk")
sns.set_palette("dark")

## Soal 1: memuat data sinusoidal

Data berisi 100 titik contoh dari satu gelombang sinus yang diberi derau, ditambah kurva acuan tanpa derau sebagai pembanding bentuk aslinya.

In [ ]:
data_path = Path("data/X_Y_Sinusoid_Data.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/X_Y_Sinusoid_Data.csv"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

sinusoid = pd.read_csv(data_path)
sinusoid.head()

In [ ]:
x_reference = np.linspace(0, 1.0, 100)
y_reference = np.sin(2 * np.pi * x_reference)

ax = sinusoid.set_index("x")["y"].plot(ls="", marker="o", label="data")
ax.plot(x_reference, y_reference, ls="--", label="real function")
ax.legend()
ax.set(xlabel="x data", ylabel="y data")
plt.show()

## Soal 2: polinomial derajat 20 tanpa regularisasi

Fitur polinomial derajat 20 membuat model sangat fleksibel. Tanpa penalti, model mengikuti setiap titik derau sehingga bentuk aslinya hilang.

In [ ]:
degree = 20
poly = PolynomialFeatures(degree)
linear_model = LinearRegression()

x_data = sinusoid[["x"]]
y_data = sinusoid["y"]

x_poly = poly.fit_transform(x_data)
linear_model.fit(x_poly, y_data)
y_pred_linear = linear_model.predict(x_poly)

plt.plot(x_data, y_data, marker="o", ls="", label="data")
plt.plot(x_reference, y_reference, ls="--", label="real function")
plt.plot(x_data, y_pred_linear, marker="^", alpha=0.5, label="prediksi polinomial")
plt.legend()
plt.gca().set(xlabel="x data", ylabel="y data")
plt.show()

## Soal 3: Ridge dan Lasso

Ridge memberi penalti L2 yang menyusutkan koefisien secara halus. Lasso memberi penalti L1 yang mendorong sebagian koefisien tepat menjadi nol. Keduanya meredam kebebasan model derajat tinggi.

In [ ]:
ridge_model = Ridge(alpha=0.001).fit(x_poly, y_data)
y_pred_ridge = ridge_model.predict(x_poly)

lasso_model = Lasso(alpha=0.0001).fit(x_poly, y_data)
y_pred_lasso = lasso_model.predict(x_poly)

plt.plot(x_data, y_data, marker="o", ls="", label="data")
plt.plot(x_reference, y_reference, ls="--", label="real function")
plt.plot(x_data, y_pred_linear, marker="^", alpha=0.5, label="linear regression")
plt.plot(x_data, y_pred_ridge, marker="^", alpha=0.5, label="ridge regression")
plt.plot(x_data, y_pred_lasso, marker="^", alpha=0.5, label="lasso regression")
plt.legend()
plt.gca().set(xlabel="x data", ylabel="y data")
plt.show()

In [ ]:
coefficient_table = pd.DataFrame({
    "linear regression": np.abs(linear_model.coef_.ravel()),
    "ridge regression": np.abs(ridge_model.coef_.ravel()),
    "lasso regression": np.abs(lasso_model.coef_.ravel()),
})
coefficient_table.describe()

Skala koefisien ketiga model berbeda jauh. Model tanpa penalti memakai koefisien raksasa untuk mengejar setiap titik derau, sedangkan kedua model berpenalti jauh lebih terkendali. Karena rentangnya terlalu jauh, sumbu ganda dipakai.

In [ ]:
colors = sns.color_palette()
ax_left = plt.axes()
ax_right = ax_left.twinx()

ax_left.plot(linear_model.coef_.ravel(), color=colors[0], marker="o", label="linear regression")
ax_right.plot(ridge_model.coef_.ravel(), color=colors[1], marker="o", label="ridge regression")
ax_right.plot(lasso_model.coef_.ravel(), color=colors[2], marker="o", label="lasso regression")

ax_left.set_ylim(-2e14, 2e14)
ax_right.set_ylim(-25, 25)

handles_left, labels_left = ax_left.get_legend_handles_labels()
handles_right, labels_right = ax_right.get_legend_handles_labels()
ax_left.legend(handles_left + handles_right, labels_left + labels_right)

ax_left.set(xlabel="coefficients", ylabel="linear regression")
ax_right.set(ylabel="ridge and lasso regression")
ax_left.set_xticks(range(len(linear_model.coef_)))
plt.show()

## Soal 4: data Ames Housing

Data ini memuat atribut rumah dan harga jualnya. Kolom kategorikal diubah menjadi kolom indikator dengan `get_dummies`.

In [ ]:
ames_path = Path("data/Ames_Housing_Sales.csv")
ames_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv"

if not ames_path.exists():
    ames_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(ames_url, ames_path)

houses = pd.read_csv(ames_path)
houses.head(10)

In [ ]:
houses = pd.get_dummies(houses, drop_first=True)
print("Jumlah kolom setelah encoding:", houses.shape[1])

In [ ]:
train, test = train_test_split(houses, test_size=0.3, random_state=42)

Beberapa fitur sangat miring. Logaritma `log1p` menarik ekor panjangnya sehingga sebarannya lebih simetris. Kolom dengan kemiringan di atas 0,75 ditransformasi, kecuali target `SalePrice`.

In [ ]:
float_columns = houses.select_dtypes(include="float64").columns
print("Jumlah kolom float:", len(float_columns))

In [ ]:
skew_limit = 0.75
skew_values = train[float_columns].skew()

skewed_columns = (
    skew_values[skew_values.abs() > skew_limit]
    .sort_values(ascending=False)
    .rename("Skew")
    .to_frame()
)
skewed_columns

In [ ]:
field = "BsmtFinSF1"
fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(10, 4))
train[field].hist(ax=ax_before)
train[field].map(np.log1p).hist(ax=ax_after)
ax_before.set(title="before log1p", ylabel="frequency", xlabel="value")
ax_after.set(title="after log1p", ylabel="frequency", xlabel="value")
fig.suptitle(f'Kolom "{field}"')
fig.tight_layout()
plt.show()

In [ ]:
for column in skewed_columns.index.tolist():
    if column == "SalePrice":
        continue
    train[column] = np.log1p(train[column])
    test[column] = np.log1p(test[column])

In [ ]:
feature_columns = [column for column in train.columns if column != "SalePrice"]

X_train = train[feature_columns]
y_train = train["SalePrice"]
X_test = test[feature_columns]
y_test = test["SalePrice"]

## Soal 5: RMSE dan regresi linear dasar

Fungsi `rmse` mengembalikan akar rata-rata kuadrat galat, sehingga satuannya sama dengan harga.

In [ ]:
def rmse(y_true, y_predicted):
    return np.sqrt(mean_squared_error(y_true, y_predicted))


baseline_model = LinearRegression().fit(X_train, y_train)
baseline_rmse = rmse(y_test, baseline_model.predict(X_test))
print("RMSE regresi linear:", baseline_rmse)

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = plt.axes()
ax.plot(y_test, baseline_model.predict(X_test), marker="o", ls="", ms=3.0)
limit = (0, y_test.max())
ax.set(xlabel="Actual Price", ylabel="Predicted Price", xlim=limit, ylim=limit,
       title="Linear Regression Results")
plt.show()

## Soal 6: RidgeCV

`RidgeCV` mencoba beberapa nilai alpha dengan validasi silang internal, lalu menyimpan alpha terbaik.

In [ ]:
alphas = [0.005, 0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 80]

ridge_cv = RidgeCV(alphas=alphas, cv=4).fit(X_train, y_train)
ridge_rmse = rmse(y_test, ridge_cv.predict(X_test))
print("alpha terbaik:", ridge_cv.alpha_, "| RMSE:", ridge_rmse)

## Soal 7: LassoCV, ElasticNetCV

`LassoCV` memilih alpha untuk penalti L1. Sebagian koefisien akan menjadi nol, sehingga model sekaligus melakukan pemilihan fitur. ElasticNet menggabungkan penalti L1 dan L2.

In [ ]:
small_alphas = np.array([1e-5, 5e-5, 0.0001, 0.0005])

lasso_cv = LassoCV(alphas=small_alphas, max_iter=int(5e4), cv=3).fit(X_train, y_train)
lasso_rmse = rmse(y_test, lasso_cv.predict(X_test))
print("alpha terbaik:", lasso_cv.alpha_, "| RMSE:", lasso_rmse)
print(f"Dari {len(lasso_cv.coef_)} koefisien, {len(lasso_cv.coef_.nonzero()[0])} tidak nol.")

In [ ]:
l1_ratios = np.linspace(0.1, 0.9, 9)

elastic_cv = ElasticNetCV(alphas=small_alphas, l1_ratio=l1_ratios, max_iter=int(1e4)).fit(X_train, y_train)
elastic_rmse = rmse(y_test, elastic_cv.predict(X_test))
print("alpha:", elastic_cv.alpha_, "| l1_ratio:", elastic_cv.l1_ratio_, "| RMSE:", elastic_rmse)

In [ ]:
rmse_table = pd.Series(
    [baseline_rmse, ridge_rmse, lasso_rmse, elastic_rmse],
    index=["Linear", "Ridge", "Lasso", "ElasticNet"],
).to_frame("RMSE")
rmse_table

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = plt.axes()

for model, label in zip([ridge_cv, lasso_cv, elastic_cv], ["Ridge", "Lasso", "ElasticNet"]):
    ax.plot(y_test, model.predict(X_test), marker="o", ls="", ms=3.0, label=label)

legend = plt.legend(frameon=True)
legend.get_frame().set_edgecolor("black")
legend.get_frame().set_linewidth(1.0)
ax.set(xlabel="Actual Price", ylabel="Predicted Price", title="Linear Regression Results")
plt.show()

## Soal 8: Stochastic Gradient Descent

`SGDRegressor` memperbarui koefisien per sebagian kecil data, bukan sekaligus. Model linear sangat peka terhadap skala fitur dan laju pembelajaran, sehingga tanpa penyesuaian hasilnya meledak.

In [ ]:
model_parameters = {
    "Linear": {"penalty": None},
    "Lasso": {"penalty": "l1", "alpha": lasso_cv.alpha_},
    "Ridge": {"penalty": "l2", "alpha": ridge_cv.alpha_},
    "ElasticNet": {"penalty": "elasticnet", "alpha": elastic_cv.alpha_, "l1_ratio": elastic_cv.l1_ratio_},
}

sgd_rmse = {}
for label, parameters in model_parameters.items():
    sgd = SGDRegressor(**parameters)
    sgd.fit(X_train, y_train)
    sgd_rmse[label] = rmse(y_test, sgd.predict(X_test))

rmse_table["RMSE-SGD"] = pd.Series(sgd_rmse)
rmse_table

Galatnya sangat besar karena algoritma divergen: laju pembelajaran bawaan terlalu tinggi untuk data yang belum diskalakan.

In [ ]:
scaled_rmse = {}
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for label, parameters in model_parameters.items():
    sgd = SGDRegressor(eta0=1e-7, **parameters)
    sgd.fit(X_train_scaled, y_train)
    scaled_rmse[label] = rmse(y_test, sgd.predict(X_test_scaled))

rmse_table["RMSE-SGD-scaled"] = pd.Series(scaled_rmse)
rmse_table

Setelah fitur diskalakan ke rentang 0 sampai 1 dan laju pembelajaran diperkecil, galat SGD turun ke tingkat yang sebanding dengan model lain. Ini menunjukkan bahwa penskalaan bukan formalitas, melainkan syarat agar gradient descent stabil.

## Ringkasan

- Polinomial derajat tinggi tanpa penalti menghasilkan overfitting berat.
- Ridge menyusutkan koefisien, Lasso juga memilih fitur dengan menolkan sebagian koefisien.
- `RidgeCV`, `LassoCV`, dan `ElasticNetCV` menyetel alpha secara otomatis lewat validasi silang.
- Gradient descent memerlukan data terskala dan laju pembelajaran yang wajar.